# 감시 카메라 이미지에서 위협 요소 탐지

군사 기지 감시 초소에서는 감시 카메라가 주기적으로 이미지를 촬영하고 있습니다. 해당 감시 카메라 시스템은 적의 침투나 이상 상황을 감지하기 위해 경고 표지판과 같은 빨간색 사각형 객체를 추적합니다. 이 문제의 목표는 주어진 감시 카메라 이미지에서 빨간색 경고 표지판(사각형 형태)을 탐지하여 그 좌표를 정확히 추출하는 프로그램을 작성하는 것입니다.

## 데이터 셋 설명

감시 카메라로 촬영된 이미지 파일(image_1.jpg, image_2.jpg, ..., image_100.jpg까지 100개의 이미지)은 data 폴더 바로 아래에 있습니다. 각 이미지에는 빨간색 사각형 객체(경고 표지판)가 1개 포함되어 있습니다.

## 문제 설명

 - 주어진 이미지에서 빨간색 사각형 경고 표지판의 좌표를 탐지하고 (left, right, top, bottom) 형식으로 출력하세요.
 - 이미지의 좌표 체계는 좌측 상단이 (0, 0)이며, x축은 오른쪽으로 증가하고, y축은 아래쪽으로 증가합니다. 모든 이미지의 가로 세로 픽셀 크기는 512x512이며, 우측 하단의 좌표는 (512, 512)입니다.
    - left, right은 x축이며 right의 값이 left보다 커야 합니다.
    - top, bottom은 y축이며 bottom의 값이 top의 값보다 커야 합니다.
 - OpenCV 라이브러리를 사용할 수 있습니다.

## 초기코드(Python)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import cv2

# 테스트 예시
for i in range(1, 101):
    filename = f"image_{i}.jpg"
    image_path = f"./drive/MyDrive/maicon24/day1/Problem#3/data/{filename}"

In [ ]:
import os
import numpy as np
import pandas as pd

# 이미지 경로: Colab(초기코드) 경로 우선, 없으면 로컬 ./data 사용
D = './drive/MyDrive/maicon24/day1/Problem#3/data/'
if not os.path.exists(D + 'image_1.jpg'):
    D = './data/'

def detect_red_box(path):
    """빨간 사각형의 (left, right, top, bottom) 반환. right/bottom은 픽셀 경계(= x+w, y+h) 기준."""
    img = cv2.imread(path)                               # BGR
    b, g, r = [img[..., k].astype(int) for k in range(3)]
    mask = ((r > 180) & (g < 80) & (b < 80)).astype(np.uint8)   # JPEG 번짐 허용한 빨강 마스크
    n, _, stats, _ = cv2.connectedComponentsWithStats(mask)
    assert n > 1, f'red object not found: {path}'
    j = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))  # 가장 큰 연결 성분 = 표지판
    x, y, w, h = stats[j, :4]
    return int(x), int(x + w), int(y), int(y + h)

rows = []
for i in range(1, 101):
    filename = f"image_{i}.jpg"
    left, right, top, bottom = detect_red_box(D + filename)
    rows.append((filename, left, right, top, bottom))
rows[:5]

### 최종 제출 파일

submission.csv라는 이름의 CSV 파일로 저장해주세요. 예를 들어 최종 제출 파일의 처음 5개(헤더 제외)의 레코드를 보면 다음과 같은 형태가 되어야 합니다.

| filename | left | right | top | bottom |
| --- | --- | --- | --- | --- |
| image_1.jpg | 217 | 277 | 85 | 152 |
| image_2.jpg | 98 | 175 | 398 | 494 |
| image_3.jpg | 92 | 182 | 418 | 498 |
| image_4.jpg | 175 | 253 | 454 | 487 |
| image_5.jpg | 268 | 363 | 235 | 273 |

In [ ]:
df = pd.DataFrame(rows, columns=['filename', 'left', 'right', 'top', 'bottom'])

# 검증: 100행, 좌표 범위/순서
assert len(df) == 100
assert ((df.left >= 0) & (df.right <= 512) & (df.top >= 0) & (df.bottom <= 512)).all()
assert ((df.right > df.left) & (df.bottom > df.top)).all()
df.head()

### 결과 저장

채점을 위해, 위에서 구한 데이터를 현재 파일과 같은 디렉토리(.ipynb 파일이 있는 디렉토리)에 `submission.csv`이라는 이름으로 저장해주세요.

In [ ]:
df.to_csv('submission.csv', index=False)
chk = pd.read_csv('submission.csv')
assert list(chk.columns) == ['filename', 'left', 'right', 'top', 'bottom'] and len(chk) == 100
chk.head()

In [ ]:
# (원본 예시 `df.to_csv('submission.csv')`는 index 열이 추가되어 제출 형식과 달라 사용하지 않음)
print(open('submission.csv').read().splitlines()[:3])